# S08: observe failure and design recovery

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/07_reliability.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

70-minute lab. A controlled injected failure is useful evidence about software behavior. It is not a measurement of provider reliability or live serving performance.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–25 min | Open the circuit and recover.** Inject two explicit dependency failures. A third attempt should stop before another model call. Advance a controllable clock past cooldown; replace the failure with an actual supported local response fixture. This fixture tests recovery mechanics only.

In [ ]:
class Clock:
    def __init__(self):self.now=0
    def __call__(self):return self.now
clock=Clock()
class FailingDependency:
    model='injected-failure-not-a-model'
    def __init__(self):self.calls=0
    def json(self,*args):self.calls+=1;raise TimeoutError('injected')
ix=Index();ix.build('v1',read_data('policies.json'));db=Purchases();failure=FailingDependency()
p=Platform(ix,db,failure,clock=clock)
q='standard supplier payment terms'
statuses=[p.ask(BUYER,'policy',q)['status'] for _ in range(3)]
assert statuses==['dependency_error','dependency_error','circuit_open'] and failure.calls==2
clock.now=11
p.model=None # Explicit recovery to evidence-only mode, not a hidden model fallback.
recovered=p.ask(BUYER,'policy',q)
assert recovered['status']=='evidence_only'
print(statuses,recovered['status']);print(p.events)

**25–45 min | Inspect attribution and SLOs.** Decide which events are user errors, admission decisions and service failures. Avoid measuring a model SLO from this injected clock. Propose request availability, supported-answer rate, p95 latency and cost per successful task as separate measures. Raw prompts and credentials are absent from these default traces.

In [ ]:
from collections import Counter
print('Observed software statuses:',dict(Counter(e['status'] for e in p.events)))
# Capacity worksheet: hypothetical values, not a load test.
arrival_rate=3; service_seconds=2.5; workers=10
in_flight=arrival_rate*service_seconds
print({'mean_in_flight_estimate':in_flight,'hypothetical_workers':workers,'utilisation_estimate':in_flight/workers})
print('Tail latency, provider quotas and burst queues still need measurement.')

**45–60 min | Change the policy.** Add per-application circuit state so one application's dependency failure does not block the other. Define whether two apps sharing one provider should share a provider-level circuit as well. Add bounded retries only for explicitly retryable cases, respecting provider guidance and the request deadline. Never retry an external write without a verified idempotency contract.

**60–70 min | Deliver an incident runbook.** Cover detect, classify, contain, communicate, recover and review. State when to serve cached authorised evidence, when to return a clarification and when to stop. Include identity/data version in cache design and a deletion/revocation strategy.

In [ ]:
ix.close();db.close()